In [2]:
# make_population_maps.py
# 茨城県 市町村人口のマップ描画用スクリプト
#
# 必要ライブラリ:
#   pip install geopandas matplotlib
#
# 実行方法 (例):
#   python make_population_maps.py
# すると:
#   - map_pop_2020.png  : 2020年の人口マップ
#   - map_growth_2000_2020.png : 2000→2020の増減率マップ
# が出力されます。

import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# ★自分の環境に合わせるパラメータ
# ==========================================
PANEL_FILE = "panel_ibaraki_ssdse_with_W.csv"   # さっきまで使ってたパネル
SHAPEFILE  = "N03-20240101_08.shp"             # 国土数値情報（茨城県を含む行政区域データ）

# マップを作る年（必要に応じて変えてOK）
LEVEL_YEAR   = 2020      # 人口レベルマップの年
BASE_YEAR    = 2000      # 増減率の基準年
TARGET_YEAR  = 2020      # 増減率の比較対象年

# 日本語フォント（Windows想定）
# もし文字化けするようならコメントアウトするか、別フォントに変えてください
plt.rcParams["font.family"] = "MS Gothic"


# ==========================================
# パネルデータの読み込み
# ==========================================
def load_panel(path: str) -> pd.DataFrame:
    print("=== Loading panel data ===")
    df = pd.read_csv(path, encoding="utf-8-sig")
    print("Panel shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head())
    return df


# ==========================================
# shapefile の読み込み & 茨城県だけ抽出
# ==========================================
def load_ibaraki_shapefile(path: str) -> gpd.GeoDataFrame:
    """
    国土数値情報 N03 (行政区域) から 茨城県の市町村ポリゴンを取り出す。
    すでに neighbor 作成で動いている想定なので、
    pref = N03_001, city = N03_004 でハードコードしている。
    """
    print("\n=== Loading shapefile ===")
    gdf = gpd.read_file(path)
    print("Shapefile columns:", gdf.columns.tolist())

    # 茨城県だけ抽出
    gdf_ibaraki = gdf[gdf["N03_001"] == "茨城県"].copy()
    print("Filtered to 茨城県:", len(gdf_ibaraki), "rows")

    # 市町村名（N03_004）を使って city_name を作成
    gdf_ibaraki["city_name"] = (
        gdf_ibaraki["N03_004"]
        .astype(str)
        .str.replace("　", "")  # 全角スペース削除（あれば）
        .str.strip()
    )

    # 同一市町村が複数ポリゴンに分かれているので、city_name で dissolve
    gdf_city = gdf_ibaraki.dissolve(by="city_name", as_index=False)
    print("After dissolve (one polygon per city):", len(gdf_city), "rows")
    print(gdf_city[["city_name"]].head())
    return gdf_city


# ==========================================
# 年度別の人口分布マップ
# ==========================================
def make_level_map(panel: pd.DataFrame,
                   gdf_city: gpd.GeoDataFrame,
                   year: int,
                   out_png: str = None):
    print(f"\n=== Making population level map for year={year} ===")

    # 指定年だけ抽出
    df_year = panel[panel["year"] == year].copy()
    df_year = df_year.groupby("city_name", as_index=False)["population"].sum()

    print("Data for map (first 5 rows):")
    print(df_year.head())

    # shapefile とマージ
    gdf_merged = gdf_city.merge(df_year, on="city_name", how="left")
    missing = gdf_merged["population"].isna().sum()
    if missing > 0:
        print(f"Warning: {missing} polygons have NaN population for year={year}")

    # プロット
    fig, ax = plt.subplots(figsize=(8, 8))
    gdf_plot = gdf_merged.plot(
        column="population",
        ax=ax,
        cmap="viridis",
        legend=True,
        edgecolor="black",
        linewidth=0.3,
    )

    # ---★ カラーバー（凡例）のフォント調整 ★---
    # geodataframe.plot(legend=True) の場合，fig.axes[1] が colorbar 軸になることが多い
    if len(fig.axes) > 1:
        cax = fig.axes[1]
        cax.tick_params(labelsize=12)  # 目盛りフォントサイズ
        cax.set_ylabel("人口", fontsize=12)

    # ---★ 主要都市ラベルの追加 ★---
    import matplotlib.patheffects as pe
    label_cities = ["水戸市", "つくば市", "日立市", "常陸太田市",
                    "ひたちなか市", "古河市", "鹿嶋市"]

    gdf_center = gdf_merged.copy()
    gdf_center["center"] = gdf_center.geometry.centroid

    for _, row in gdf_center[gdf_center["city_name"].isin(label_cities)].iterrows():
        x = row["center"].x
        y = row["center"].y
        name = row["city_name"]
        ax.text(
            x, y, name,
            fontsize=10,
            ha="center", va="center",
            path_effects=[pe.withStroke(linewidth=1, foreground="white")]
        )

    # タイトル（表題）のフォントサイズを大きめに
    ax.set_title(f"茨城県 市町村人口（{year}年）", fontsize=20)
    ax.axis("off")

    if out_png is None:
        out_png = f"map_pop_{year}.png"

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    print(f"Saved population level map to {out_png}")
    plt.close(fig)


# ==========================================
# 2時点の人口増減率マップ
# ==========================================
def make_growth_map(panel: pd.DataFrame,
                    gdf_city: gpd.GeoDataFrame,
                    base_year: int,
                    target_year: int,
                    out_png: str = None):
    print(f"\n=== Making growth map: {base_year} -> {target_year} ===")

    # 基準年
    df_base = panel[panel["year"] == base_year][["city_name", "population"]].copy()
    df_base = df_base.groupby("city_name", as_index=False)["population"].sum()
    df_base = df_base.rename(columns={"population": "pop_base"})

    # 比較年
    df_target = panel[panel["year"] == target_year][["city_name", "population"]].copy()
    df_target = df_target.groupby("city_name", as_index=False)["population"].sum()
    df_target = df_target.rename(columns={"population": "pop_target"})

    # マージして増減率を計算
    df_merge = pd.merge(df_base, df_target, on="city_name", how="inner")
    df_merge["growth_rate"] = (
        df_merge["pop_target"] - df_merge["pop_base"]
    ) / df_merge["pop_base"]

    print("Growth data (first 5 rows):")
    print(df_merge.head())

    # shapefile と結合
    gdf_merged = gdf_city.merge(df_merge, on="city_name", how="left")
    missing = gdf_merged["growth_rate"].isna().sum()
    if missing > 0:
        print(f"Warning: {missing} polygons have NaN growth_rate for {base_year}->{target_year}")

    # プロット
    fig, ax = plt.subplots(figsize=(8, 8))
    vmin = gdf_merged["growth_rate"].quantile(0.05)
    vmax = gdf_merged["growth_rate"].quantile(0.95)
    absmax = max(abs(vmin), abs(vmax))

    gdf_plot = gdf_merged.plot(
        column="growth_rate",
        ax=ax,
        cmap="bwr",              # blue-white-red
        legend=True,
        edgecolor="black",
        linewidth=0.3,
        vmin=-absmax,
        vmax=absmax,
    )

    # ---★ カラーバー（凡例）のフォント調整 ★---
    if len(fig.axes) > 1:
        cax = fig.axes[1]
        cax.tick_params(labelsize=12)
        cax.set_ylabel("人口増減率", fontsize=12)

    # ---★ 主要都市ラベルの追加 ★---
    import matplotlib.patheffects as pe
    label_cities = ["水戸市", "つくば市", "日立市", "常陸太田市",
                    "ひたちなか市", "古河市", "鹿嶋市"]

    gdf_center = gdf_merged.copy()
    gdf_center["center"] = gdf_center.geometry.centroid

    for _, row in gdf_center[gdf_center["city_name"].isin(label_cities)].iterrows():
        x = row["center"].x
        y = row["center"].y
        name = row["city_name"]
        ax.text(
            x, y, name,
            fontsize=10,
            ha="center", va="center",
            path_effects=[pe.withStroke(linewidth=1, foreground="white")]
        )

    ax.set_title(
        f"茨城県 市町村人口増減率（{base_year}→{target_year}）",
        fontsize=20
    )
    ax.axis("off")

    if out_png is None:
        out_png = f"map_growth_{base_year}_{target_year}.png"

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    print(f"Saved growth map to {out_png}")
    plt.close(fig)

# ==========================================
# main
# ==========================================
def main():
    # 1) パネルデータ読み込み
    if not os.path.exists(PANEL_FILE):
        raise FileNotFoundError(f"{PANEL_FILE} が見つかりません。")
    panel = load_panel(PANEL_FILE)

    # 2) shapefile 読み込み
    if not os.path.exists(SHAPEFILE):
        raise FileNotFoundError(f"{SHAPEFILE} が見つかりません。")
    gdf_city = load_ibaraki_shapefile(SHAPEFILE)

    # 3) 単年の人口分布マップ
    make_level_map(panel, gdf_city, year=LEVEL_YEAR)

    # 4) 2時点の人口増減率マップ
    make_growth_map(panel, gdf_city, base_year=BASE_YEAR, target_year=TARGET_YEAR)


if __name__ == "__main__":
    main()


=== Loading panel data ===
Panel shape: (2068, 23)
Columns: ['city_name', 'year', 'population', 'log_pop', 'log_pop_lag1', 'city_code', 'A1301', 'A1302', 'A1303', 'A4101', 'A4200', 'A5101', 'A5102', 'A7101', 'A710101', 'share_under15', 'share_over65', 'birth_rate', 'death_rate', 'net_mig_rate', 'nat_inc_rate', 'soc_inc_rate', 'W_log_pop_lag1']
  city_name  year  population    log_pop  log_pop_lag1  city_code   A1301  \
0   かすみがうら市  1976     36119.0  10.494574     10.485312        NaN  4376.0   
1   かすみがうら市  1977     36736.0  10.511512     10.494574        NaN  4376.0   
2   かすみがうら市  1978     37387.0  10.529078     10.511512        NaN  4376.0   
3   かすみがうら市  1979     37919.0  10.543208     10.529078        NaN  4376.0   
4   かすみがうら市  1980     38797.0  10.566098     10.543208        NaN  4376.0   

     A1302    A1303  A4101  ...    A7101  A710101  share_under15  \
0  22859.0  12779.0  176.0  ...  15271.0  15230.0       0.121155   
1  22859.0  12779.0  176.0  ...  15271.0  15230.0      

C:\Users\johns\AppData\Local\Temp\ipykernel_22912\4239745043.py:125: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_center["center"] = gdf_center.geometry.centroid


Saved population level map to map_pop_2020.png

=== Making growth map: 2000 -> 2020 ===
Growth data (first 5 rows):
  city_name  pop_base  pop_target  growth_rate
0   かすみがうら市   45229.0     40087.0    -0.113688
1   つくばみらい市   40532.0     49872.0     0.230435
2      つくば市  191814.0    241656.0     0.259845
3    ひたちなか市  151673.0    156581.0     0.032359
4       下妻市   46544.0     42521.0    -0.086434


C:\Users\johns\AppData\Local\Temp\ipykernel_22912\4239745043.py:215: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_center["center"] = gdf_center.geometry.centroid


Saved growth map to map_growth_2000_2020.png
